In [8]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
from pathlib import Path

2.21.0


In [11]:
DATA_DIR = "data_uploaded/"

for L in [10, 20, 30, 40, 60]:

    data = np.load(f"{DATA_DIR}/L{L}_ising.npz")
    """split data into input and output"""
    T = data["temperatures"]
    T_c = 2 / np.log(1 + np.sqrt(2))        
    labels = np.transpose(np.array([(T > T_c).astype(int), (T < T_c).astype(int)]))    #create labels from temperature
    configs = data["spins"]

    rng = np.random.default_rng(seed=42)
    idx = np.arange(len(T))
    rng.shuffle(idx)

    # Apply the same permutation to all arrays
    T = T[idx]
    configs = configs[idx]
    labels  = labels[idx, :]
    print(labels.shape)

    """split into training, validation and test data"""
    train_conf, val_conf = np.split(configs, [80000])
    train_label, val_label = np.split(labels, [80000], axis=0)
    train_T, val_T = np.split(T, [80000])
    print(train_conf.shape)

(110250, 2)
(80000, 100)
(110250, 2)
(80000, 400)
(110250, 2)
(80000, 900)
(110250, 2)
(80000, 1600)
(110250, 2)
(80000, 3600)


In [13]:
DATA_DIR = "data_uploaded/"

BATCH_SIZE = 32

def build_model(config_size, hidden_nodes, l2):
    initializer = tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0)
    glorot_uniform = tf.keras.initializers.GlorotUniform(seed=42)
    x = tf.keras.Input((config_size,))
    y = tf.keras.layers.Dense(hidden_nodes, activation='sigmoid', kernel_initializer=initializer, kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    #k = tf.keras.layers.Dropout(0.3)(y)
    z = tf.keras.layers.Dense(2, activation='sigmoid')(y)
    #y = tf.keras.layers.Dense(hidden_nodes, activation='relu',
    #                      kernel_initializer='glorot_uniform',
    #                      kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    #z = tf.keras.layers.Dense(2, activation='softmax')(y)
    model = tf.keras.Model(inputs=x, outputs=z)
    #model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.summary()
    return model

for L in [10, 20, 30, 40, 60]:

    data = np.load(f"{DATA_DIR}/L{L}_ising.npz")
    """split data into input and output"""
    T = data["temperatures"]
    T_c = 2 / np.log(1 + np.sqrt(2))        
    labels = np.transpose(np.array([(T > T_c).astype(int), (T < T_c).astype(int)]))    #create labels from temperature
    configs = data["spins"]

    rng = np.random.default_rng(seed=42)
    idx = np.arange(len(T))
    rng.shuffle(idx)

    # Apply the same permutation to all arrays
    T = T[idx]
    configs = configs[idx]
    labels  = labels[idx, :]
    print(labels.shape)

    """split into training, validation and test data"""
    train_conf, val_conf = np.split(configs, [90000])
    train_label, val_label = np.split(labels, [90000], axis=0)
    train_T, val_T = np.split(T, [90000])
    #print(train_conf.shape)

    l2 = 0.05   # regularization strength λ

    model3_2 = build_model(configs.shape[1], 100, l2)

    w_init, b_init = model3_2.layers[1].get_weights()

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=8, min_lr=1e-6)
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

    history = model3_2.fit(
        train_conf,
        train_label,
        validation_data = (val_conf, val_label),
        batch_size = 128,
        epochs = 75,
        callbacks=[reduce_lr, early_stop]
    )
    #callbacks=[reduce_lr, early_stop]


    file_path = f'models_100_op/training_history_100_op/L{L}.json'
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, 'w') as f:
        json.dump(history.history, f)

    print(f"Successfully saved history to {file_path}")

    # Save after training
    model3_2.save(f"models_100_op/ising_classifier_L{L}.h5")


(110250, 2)


Model: "functional_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_26 (InputLayer)     │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 2)              │           202 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,302 (40.24 KB)

 Trainable params: 10,302 (40.24 KB)

...

 Total params: 360,302 (1.37 MB)

 Trainable params: 360,302 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10850.3008 - val_loss: 5977.1802 - learning_rate: 0.0010
Epoch 2/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3634.7151 - val_loss: 1979.9169 - learning_rate: 0.0010
Epoch 3/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1177.9185 - val_loss: 616.6276 - learning_rate: 0.0010
Epoch 4/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 353.8984 - val_loss: 174.4127 - learning_rate: 0.0010
Epoch 5/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 95.4807 - val_loss: 43.4843 - learning_rate: 0.0010
Epoch 6/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 22.5174 - val_loss: 9.3340 - learning_rate: 0.0010
Epoch 7/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4.5566 - val_loss: 1.7084 - learning_rate: 0.0010
Epoch 8/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7981 - val_loss: 0.2795 - learning_rate: 0.0010
Epoch 9/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1401 - val_loss: 0.0644 - learnin

Successfully saved history to models_100_op/training_history_100_op/L60.json
